In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

CATALOG         = "netflix_catalog"
SCHEMA          = "net_schema"
bronze_path     = "abfss://bronze@mynetflixproject.dfs.core.windows.net/netflix_titles"
silver_path     = "abfss://silver@mynetflixproject.dfs.core.windows.net/netflix_titles"
quarantine_path = "abfss://silver@mynetflixproject.dfs.core.windows.net/_quarantine/netflix_titles"
checkpoint_path = "abfss://silver@mynetflixproject.dfs.core.windows.net/_checkpoints/netflix_titles"

def process_silver_microbatch(micro_df, batch_id):
    if micro_df.isEmpty():
        return

    if "_rescued_data" not in micro_df.columns:
        micro_df = micro_df.withColumn("_rescued_data", F.lit(None).cast("string"))

    # --- 1. Standardise Strings ---
    staged_df = (
        micro_df
        .withColumn("show_id", F.trim(F.col("show_id").cast("string")))
        .withColumn(
            "type",
            F.when(F.lower(F.trim(F.col("type"))) == "movie", F.lit("Movie"))
             .when(F.lower(F.trim(F.col("type"))) == "tv show", F.lit("TV Show"))
             .otherwise(F.trim(F.col("type")))
        )
        .withColumn(
            "title",
            F.when(F.col("title").isNull() | (F.trim(F.col("title")) == ""), F.lit(None).cast("string"))
             .otherwise(F.trim(F.regexp_replace(F.col("title"), r"\s+", " ")))
        )
        .withColumn(
            "rating",
            F.when(F.col("rating").isNull() | (F.trim(F.col("rating")) == ""), F.lit("Not Rated"))
             .otherwise(F.regexp_replace(F.upper(F.trim(F.col("rating"))), r"^TV\s+", "TV-"))
        )
        .withColumn(
            "description",
            F.when(F.col("description").isNull() | (F.trim(F.col("description")) == ""), F.lit(None).cast("string"))
             .otherwise(F.trim(F.regexp_replace(F.col("description"), r"\s+", " ")))
        )
    )

    # --- 2. Validation Rules ---
    mins_int    = F.col("duration_minutes").cast("int")
    seasons_int = F.col("duration_seasons").cast("int")
    year_int    = F.col("release_year").cast("int")
    parsed_date = F.coalesce(
        F.to_date(F.trim(F.col("date_added")), "M/d/yyyy"),
        F.to_date(F.trim(F.col("date_added")), "MMMM d, yyyy")
    )
    valid_ratings = [
        "TV-MA", "TV-14", "TV-PG", "TV-Y", "TV-Y7", "TV-Y7-FV", "TV-G",
        "R", "PG-13", "PG", "G", "NC-17", "NR", "UR", "Not Rated"
    ]

    invalid_condition = (
        F.col("_rescued_data").isNotNull() |
        F.col("show_id").isNull() | (F.col("show_id") == "") | ~F.col("show_id").rlike(r"^[0-9]+$") |
        F.col("type").isNull() | ~F.col("type").isin("Movie", "TV Show") |
        F.col("title").isNull() |
        ((F.col("type") == "Movie") & (mins_int.isNull() | (mins_int <= 0))) |
        ((F.col("type") == "TV Show") & F.col("duration_minutes").isNotNull()) |
        ((F.col("type") == "TV Show") & (seasons_int.isNull() | (seasons_int <= 0))) |
        ((F.col("type") == "Movie") & F.col("duration_seasons").isNotNull()) |
        (F.col("date_added").isNotNull() & (F.trim(F.col("date_added")) != "") & parsed_date.isNull()) |
        year_int.isNull() | (year_int < 1920) | (year_int > F.year(F.current_date()) + 1) |
        ~F.col("rating").isin(valid_ratings)
    )

    # --- 3. Split Quarantine & Clean ---
    quarantine_df = (
        staged_df.filter(invalid_condition)
        .withColumn(
            "quarantine_reason",
            F.when(F.col("_rescued_data").isNotNull(), "Corrupt CSV / Rescued Data")
             .when(F.col("show_id").isNull() | (F.col("show_id") == "") | ~F.col("show_id").rlike(r"^[0-9]+$"), "Invalid or Missing show_id")
             .when(F.col("type").isNull() | ~F.col("type").isin("Movie", "TV Show"), "Invalid or Missing type")
             .when(F.col("title").isNull(), "Missing title")
             .when((F.col("type") == "Movie") & (mins_int.isNull() | (mins_int <= 0)), "Invalid Movie duration_minutes")
             .when((F.col("type") == "TV Show") & F.col("duration_minutes").isNotNull(), "TV Show has duration_minutes")
             .when((F.col("type") == "TV Show") & (seasons_int.isNull() | (seasons_int <= 0)), "Invalid TV Show duration_seasons")
             .when((F.col("type") == "Movie") & F.col("duration_seasons").isNotNull(), "Movie has duration_seasons")
             .when(F.col("date_added").isNotNull() & (F.trim(F.col("date_added")) != "") & parsed_date.isNull(), "Unparseable date_added")
             .when(year_int.isNull() | (year_int < 1920) | (year_int > F.year(F.current_date()) + 1), "Invalid or out-of-range release_year")
             .when(~F.col("rating").isin(valid_ratings), "Invalid rating value")
             .otherwise("Unknown validation failure")
        )
        .withColumn("_quarantined_at", F.current_timestamp())
    )

    clean_df = (
        staged_df.filter(~invalid_condition)
        .withColumn("duration_minutes", mins_int)
        .withColumn("duration_seasons", seasons_int)
        .withColumn("date_added", parsed_date)
        .withColumn("release_year", year_int)
        .withColumn(
            "is_added_before_release",
            F.when(F.year(F.col("date_added")) < F.col("release_year"), F.lit(True)).otherwise(F.lit(False))
        )
        .dropDuplicates(["show_id"])
        .dropDuplicates(["title", "type", "release_year"])
        .drop("_rescued_data")
        .withColumn("_silver_processed_at", F.current_timestamp())
    )

    # --- 4. Write Only New Batch to Silver & Quarantine ---
    if not clean_df.isEmpty():
        if DeltaTable.isDeltaTable(spark, silver_path):
            (
                DeltaTable.forPath(spark, silver_path).alias("target")
                .merge(clean_df.alias("source"), "target.show_id = source.show_id")
                .whenMatchedUpdateAll()
                .whenNotMatchedInsertAll()
                .execute()
            )
        else:
            clean_df.write.format("delta").mode("overwrite").save(silver_path)

    if not quarantine_df.isEmpty():
        quarantine_df.write.format("delta").mode("append").option("mergeSchema", "true").save(quarantine_path)

# Read ONLY new commits from Bronze Delta using Structured Streaming
query = (
    spark.readStream
    .format("delta")
    .load(bronze_path)
    .writeStream
    .foreachBatch(process_silver_microbatch)
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .start()
)

query.awaitTermination()

# Register Silver table in Unity Catalog so downstream tasks can read it
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {CATALOG}.{SCHEMA}.netflix_titles
    USING DELTA
    LOCATION '{silver_path}'
""")

print("✅ Incremental Silver processing complete for netflix_titles.")
